In [2]:
# ============================================
# GOLD LAYER - Star Schema
# ============================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os

spark = SparkSession.builder \
    .appName("Ecommerce-Analysis-Layer") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark Session Started Successfully")

Spark Session Started Successfully


In [7]:
fact_order_items.createOrReplaceTempView("fact_order_items")
dim_customers.createOrReplaceTempView("dim_customers")
dim_products.createOrReplaceTempView("dim_products")
dim_sellers.createOrReplaceTempView("dim_sellers")

In [6]:
fact_order_items  = spark.read.parquet("gold/fact_order_items")
dim_customers  = spark.read.parquet("gold/dim_customers")
dim_products  = spark.read.parquet("gold/dim_products")
dim_sellers  = spark.read.parquet("gold/dim_sellers")

In [9]:
#How much revenue has the platform generated?
result1 = spark.sql("""
SELECT
    ROUND(SUM(price),2) AS total_revenue
FROM fact_order_items;

""")
result1.show()

+-------------+
|total_revenue|
+-------------+
| 1.35916437E7|
+-------------+



In [11]:
#How has revenue changed month over month?
result2 = spark.sql("""
SELECT
    YEAR(order_purchase_timestamp) AS year,
    MONTH(order_purchase_timestamp) AS month,
    ROUND(SUM(price),2) AS revenue
FROM fact_order_items
GROUP BY YEAR(order_purchase_timestamp),
         MONTH(order_purchase_timestamp)
ORDER BY year,month;
""")
result2.show()



+----+-----+----------+
|year|month|   revenue|
+----+-----+----------+
|2016|    9|    267.36|
|2016|   10|  49507.66|
|2016|   12|      10.9|
|2017|    1| 120312.87|
|2017|    2| 247303.02|
|2017|    3|  374344.3|
|2017|    4| 359927.23|
|2017|    5| 506071.14|
|2017|    6|  433038.6|
|2017|    7| 498031.48|
|2017|    8| 573971.68|
|2017|    9| 624401.69|
|2017|   10| 664219.43|
|2017|   11|1010271.37|
|2017|   12| 743914.17|
|2018|    1| 950030.36|
|2018|    2| 844178.71|
|2018|    3| 983213.44|
|2018|    4| 996647.75|
|2018|    5| 996517.68|
+----+-----+----------+
only showing top 20 rows


In [12]:
#Average Order Value (CTE)?
result3 = spark.sql("""
WITH order_totals AS (

SELECT
    order_id,
    SUM(price) AS order_value

FROM fact_order_items

GROUP BY order_id

)

SELECT
    ROUND(AVG(order_value),2) AS average_order_value
FROM order_totals;
""")
result3.show()



+-------------------+
|average_order_value|
+-------------------+
|             137.75|
+-------------------+



In [14]:
#top 10 products by revenue
result4 = spark.sql("""
SELECT
    product_id,
    ROUND(SUM(price),2) AS revenue
FROM fact_order_items
GROUP BY product_id
ORDER BY revenue DESC
LIMIT 10;
""")
result4.show()



+--------------------+--------+
|          product_id| revenue|
+--------------------+--------+
|bb50f2e236e5eea01...| 63885.0|
|6cdd53843498f9289...| 54730.2|
|d6160fb7873f18409...|48899.34|
|d1c427060a0f73f6b...|47214.51|
|99a4788cb24856965...|43025.56|
|3dd2a17168ec895c7...| 41082.6|
|25c38557cf793876c...|38907.32|
|5f504b3a1c75b73d6...| 37733.9|
|53b36df67ebb7c415...|37683.42|
|aca2eb7d00ea1a7b8...| 37608.9|
+--------------------+--------+



In [15]:
#revenue by product catagory 
result5 = spark.sql("""
SELECT
    p.product_category_name_en,
    ROUND(SUM(f.price),2) AS revenue
FROM fact_order_items f
JOIN dim_products p
ON f.product_id = p.product_id
GROUP BY
    p.product_category_name_en
ORDER BY revenue DESC;
""")
result5.show()



+------------------------+----------+
|product_category_name_en|   revenue|
+------------------------+----------+
|           health_beauty|1258681.34|
|           watches_gifts|1205005.68|
|          bed_bath_table|1036988.68|
|          sports_leisure| 988048.97|
|    computers_accesso...| 911954.32|
|         furniture_decor| 729762.49|
|              cool_stuff| 635290.85|
|              housewares| 632248.66|
|                    auto| 592720.11|
|            garden_tools| 485256.46|
|                    toys|  483946.6|
|                    baby| 411764.89|
|               perfumery| 399124.87|
|               telephony| 323667.53|
|        office_furniture|  273960.7|
|              stationery| 230943.23|
|               computers| 222963.13|
|                pet_shop| 214315.41|
|     musical_instruments| 191498.88|
|        small_appliances| 190648.58|
+------------------------+----------+
only showing top 20 rows


In [16]:
#revenue by customer state
result5 = spark.sql("""
SELECT
    c.customer_state,
    ROUND(SUM(f.price),2) AS revenue
FROM fact_order_items f
JOIN dim_customers c
ON f.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY revenue DESC;
""")
result5.show()



+--------------+----------+
|customer_state|   revenue|
+--------------+----------+
|            SP|5202955.05|
|            RJ|1824092.67|
|            MG|1585308.03|
|            RS| 750304.02|
|            PR| 683083.76|
|            SC| 520553.34|
|            BA| 511349.99|
|            DF| 302603.94|
|            GO| 294591.95|
|            ES| 275037.31|
|            PE| 262788.03|
|            CE| 227254.71|
|            PA| 178947.81|
|            MT| 156453.53|
|            MA| 119648.22|
|            MS| 116812.64|
|            PB| 115268.08|
|            PI|  86914.08|
|            RN|  83034.98|
|            AL|  80314.81|
+--------------+----------+
only showing top 20 rows


In [17]:
#top customer by revenue 
result5 = spark.sql("""
SELECT
    customer_id,
    ROUND(SUM(price),2) AS revenue
FROM fact_order_items
GROUP BY customer_id
ORDER BY revenue DESC
LIMIT 10;
""")
result5.show()



+--------------------+-------+
|         customer_id|revenue|
+--------------------+-------+
|1617b1357756262bf...|13440.0|
|ec5b2ba62e5743423...| 7160.0|
|c6e2731c5b391845f...| 6735.0|
|f48d464a0baaea338...| 6729.0|
|3fd6777bbce08a352...| 6499.0|
|05455dfa7cd02f13d...| 5934.6|
|df55c14d1476a9a34...| 4799.0|
|24bbf5fd2f2e1b359...| 4690.0|
|e0a2412720e9ea4f2...| 4599.9|
|3d979689f636322c6...| 4590.0|
+--------------------+-------+



In [18]:
#avg delevary time 
result5 = spark.sql("""
SELECT
    ROUND(AVG(delivery_days),2) AS average_delivery_days
FROM fact_order_items;
""")
result5.show()



+---------------------+
|average_delivery_days|
+---------------------+
|                12.41|
+---------------------+



In [19]:
#late delevary percentage
result5 = spark.sql("""
SELECT
    ROUND(
        100.0 * SUM(CASE WHEN is_late = TRUE THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS late_delivery_percentage
FROM fact_order_items;
""")
result5.show()



+------------------------+
|late_delivery_percentage|
+------------------------+
|                    7.74|
+------------------------+



In [20]:
#Ranking sellers by revenue 
result5 = spark.sql("""
WITH seller_revenue AS (

SELECT
    seller_id,
    SUM(price) AS revenue

FROM fact_order_items

GROUP BY seller_id

)

SELECT
    seller_id,
    revenue,
    RANK() OVER(ORDER BY revenue DESC) AS seller_rank
FROM seller_revenue;
""")
result5.show()



+--------------------+------------------+-----------+
|           seller_id|           revenue|seller_rank|
+--------------------+------------------+-----------+
|4869f7a5dfa277a7d...| 229472.6300000005|          1|
|53243585a1d6dc264...|222776.05000000002|          2|
|4a3ca9315b744ce9f...| 200472.9200000013|          3|
|fa1c13f2614d7b5c4...|194042.03000000038|          4|
|7c67e1448b00f6e96...|         187923.89|          5|
|7e93a43ef30c4f03f...|         176431.87|          6|
|da8622b14eb17ae28...|160236.57000000114|          7|
|7a67c85e85bb2ce85...|141745.53000000032|          8|
|1025f0e2d44d7041d...|138968.55000000022|          9|
|955fee9216a65b617...|135171.70000000024|         10|
|46dc3b2cc0980fb8e...|128111.19000000028|         11|
|6560211a19b47992c...|         123304.83|         12|
|620c87c171fb2a6dd...|114774.50000000041|         13|
|7d13fca1522535862...|113628.97000000007|         14|
|5dceca129747e92ff...|112155.53000000013|         15|
|1f50f920176fa81da...|106939

In [21]:
#top 3 products with catagory 
result5 = spark.sql("""
WITH ranked_products AS (

SELECT
    p.product_category_name_en,
    f.product_id,
    SUM(f.price) AS revenue,
    ROW_NUMBER() OVER(
        PARTITION BY p.product_category_name_en
        ORDER BY SUM(f.price) DESC
    ) AS rn

FROM fact_order_items f

JOIN dim_products p
ON f.product_id = p.product_id

GROUP BY
    p.product_category_name_en,
    f.product_id

)

SELECT *
FROM ranked_products
WHERE rn <= 3;
""")
result5.show()



+------------------------+--------------------+------------------+---+
|product_category_name_en|          product_id|           revenue| rn|
+------------------------+--------------------+------------------+---+
|                    NULL|5a848e4ab52fd5445...|24229.029999999973|  1|
|                    NULL|eed5cbd74fac3bd79...|            9945.0|  2|
|                    NULL|b1d207586fca400a2...|            7152.0|  3|
|    agro_industry_and...|11250b0d4b709fee9...|            9111.0|  1|
|    agro_industry_and...|423a6644f0aa529e8...|            8043.0|  2|
|    agro_industry_and...|672e757f331900b9d...|            6885.0|  3|
|        air_conditioning|12485f9cdebb6ca17...|           3899.91|  1|
|        air_conditioning|83ca77d87b187321f...|2469.8500000000004|  2|
|        air_conditioning|0e34187d4312b97b5...|            2193.0|  3|
|                     art|4fe644d766c7566db...|10803.719999999996|  1|
|                     art|1bdf5e6731585cf01...|            6499.0|  2|
|     